In [ ]:
import numpy as np
import torch

In [ ]:
from test_functions import Satellite
from gp import train_multitask_gp
from acquisition import multitask_acquisition, optimize_acquisition, entropy
from active_learning import active_learning_loop

filename = "hist.pt"
x_input = [1. ,2. ,0.9, 1.5, 0.]

sat_prob = Satellite()
result = train_multitask_gp(sat_prob, num_train=10, seed=1111) 
active_learning_loop(result,
                              acq_method = 'entropy', 
                              maxiters = 90, disp = True, 
                              save_hist = (x_input, filename),
                              log_hyperparams = True)
# Review comment

## It is somewhat awkward to separately having to call active_learning_loop() like this. I suggest the following:
## sat_prob = Satellite() # instantiates the test problem
## gpmodel = train_multitask_gp(sat_prob, num_train=10, seed=1111) # initial GP model
## result = active_learning_loop(gpmodel, ...)
## Then, result has everything you need to post process.


# Review comment
The second time I ran your code, I get the following error:

```
---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
Cell In[5], line 9
      6 filename = "hist.pt"
      7 x_input = [1. ,2. ,0.9, 1.5, 0.]
----> 9 sat_prob = Satellite()
     10 result = train_multitask_gp(sat_prob, num_train=10, seed=1111) 
     11 active_learning_loop(result,
     12                               acq_method = 'entropy', 
     13                               maxiters = 90, disp = True, 
     14                               save_hist = (x_input, filename),
     15                               log_hyperparams = True)

TypeError: Can't instantiate abstract class Satellite with abstract method from_OpenMDAO
```

In [ ]:
# Review comment
## Your hist.pt never got saved after I ran your code
## I am not sure why you are loading the old hist.pt and plotting it below.

import torch
import matplotlib.pyplot as plt

fail_index = [] # what is fail index?

history = torch.load("hist.pt")
num_evals = history["num_evals"]
dist_history = history["dist_history"] 

fig = plt.figure(figsize=(12,4))

host = fig.add_subplot(121)
plt.plot(num_evals, dist_history)
for ind in fail_index:
    plt.plot([20+2*ind, 20+2*ind], [0,1.75], ':k') # not sure what you are doing here.
host.set_title('Convergence history: intersection distance vs residual evals')
host.set_xlabel('Individual residual evals')
host.set_ylabel('distance from openMDAO result')

host = fig.add_subplot(122)
plt.semilogy(num_evals, dist_history)
for ind in fail_index:
    plt.plot([20+2*ind, 20+2*ind], [0.0001, 1], ':k')
host.set_xlabel('Individual residual evals')
host.set_ylabel('distance from openMDAO result')

In [ ]:
## Review comment
# Please avoid hardcoding!


import matplotlib.pyplot as plt
import torch
from botorch.utils.transforms import normalize

mt_model = result.model
y = result.train_y
bounds = sat_prob.bounds

input_vec = torch.tensor(x_input)
npts = 40
# xvec is u12, yvec is u21
xvec, yvec = torch.meshgrid(torch.linspace(6.,12.,npts), torch.linspace(6.,20.,npts), indexing='ij')

# Analytic
sat_prob.set_vars(torch.column_stack([input_vec.repeat(npts**2,1),xvec.reshape(-1,1),yvec.reshape(-1,1)]))

r1vec = sat_prob.res[:,0]
r2vec = sat_prob.res[:,1]

# Predictions
test_x = torch.column_stack([input_vec.repeat(npts**2,1),
                             xvec.reshape(-1,1),
                             yvec.reshape(-1,1)])
test_x = normalize(test_x,bounds)
test_x1 = torch.column_stack([test_x, torch.ones(npts**2,1)*0]) # task id1
test_x2 = torch.column_stack([test_x, torch.ones(npts**2,1)*1]) # task id2
# test_x1.requires_grad = True

# Write a function unstandardize() to do this.
prediction1 = y.mean().item()+(mt_model.likelihood(mt_model(test_x1)))*y.std().item()
prediction2 = y.mean().item()+(mt_model.likelihood(mt_model(test_x2)))*y.std().item()

fig = plt.figure(figsize=(18,6))

# r1
ax1 = fig.add_subplot(131)
cf1 = ax1.contourf(xvec, yvec, prediction1.mean.detach().numpy().reshape(40,40),
                  cmap = 'plasma')
ax1.contour(xvec,yvec,prediction1.mean.detach().numpy().reshape(40,40), 
            levels=[0.], 
            linestyles='-', linewidths=3, colors='k')
ax1.contour(xvec, yvec, prediction1.mean.detach().reshape(40, 40) + 2.* prediction1.stddev.detach().reshape(40, 40), levels=[0.], linestyles='--', linewidths=1, colors='k')
ax1.contour(xvec, yvec, prediction1.mean.detach().reshape(40, 40) - 2.* prediction1.stddev.detach().reshape(40, 40), levels=[0.], linestyles='--', linewidths=1, colors='k')
# ax1.scatter(train_x1[...,-2],train_x1[...,-1],c='w',s=15)
fig.colorbar(cf1)
ax1.set_title(r'$r_1$')
ax1.set_xlabel(r'$u_{12}$')
ax1.set_ylabel(r'$u_{21}$')

# r2
ax2 = fig.add_subplot(132)
cf2 = ax2.contourf(xvec, yvec, prediction2.mean.detach().numpy().reshape(40,40),
                  cmap = 'plasma')
ax2.contour(xvec,yvec,prediction2.mean.detach().numpy().reshape(40,40), 
            levels=[0.], 
            linestyles='-', linewidths=3, colors='k')
ax2.contour(xvec, yvec, prediction2.mean.detach().reshape(40, 40) + 2.* prediction2.stddev.detach().reshape(40, 40), levels=[0.], linestyles='--', linewidths=1, colors='k')
ax2.contour(xvec, yvec, prediction2.mean.detach().reshape(40, 40) - 2.* prediction2.stddev.detach().reshape(40, 40), levels=[0.], linestyles='--', linewidths=1, colors='k')
# ax2.scatter(train_x2[...,-2],train_x2[...,-1],c='w',s=15)
fig.colorbar(cf2)
ax2.set_title(r'$r_2$')
ax2.set_xlabel(r'$u_{12}$')
ax2.set_ylabel(r'$u_{21}$')

# both
ax3 = fig.add_subplot(133)
# cf3 = ax3.contourf(xvec,yvec,prediction3.mean.detach().numpy().reshape(40,40),
#              cmap = 'plasma')
# fig.colorbar(cf3)
s1 = ax3.contour(xvec,yvec,r1vec.reshape(40,40),
            levels=[0.],
            linestyles='--', linewidths=3, colors='r')
s1_artist,_ = s1.legend_elements()
ax3.contour(xvec,yvec,r2vec.reshape(40,40),
            levels=[0.],
            linestyles='--', linewidths=3, colors='r')
s2 = ax3.contour(xvec,yvec,prediction1.mean.detach().numpy().reshape(40,40),
            levels=[0.],
            linestyles='-', linewidths=1.5, colors='k')
s2_artist,_ = s2.legend_elements()
s3 = ax3.contour(xvec,yvec,prediction2.mean.detach().numpy().reshape(40,40),
            levels=[0.],
            linestyles='-', linewidths=1.5, colors='b')
s3_artist,_ = s3.legend_elements()
ax3.set_title(r'$r_1$,$r_2$')
ax3.set_xlabel(r'$u_{12}$')
ax3.set_ylabel(r'$u_{21}$')
#ax3.legend([s1_artist, s3_artist], ['Analytic Residuals','Model Residuals'])
ax3.legend([s1_artist[0], s2_artist[0], s3_artist[0]],['Analytic Residuals','Model Predicted $r_1$','Model Predicted $r_2$'])

In [ ]:
from test_functions import Satellite
from gp import train_multitask_gp
from acquisition import multitask_acquisition, optimize_acquisition, entropy
from active_learning import active_learning_loop

x_input = torch.rand(10,5)

for i in range(0,20):
    print("hist"+str(i))
    filename = "hist" + str(i) + ".pt"
    sat_prob = Satellite()
    result = train_multitask_gp(sat_prob, num_train=10, seed=1111)
    active_learning_loop(result,
                         acq_method = 'entropy', 
                         maxiters = 30, disp = True, 
                         save_hist = (x_input, filename))



In [ ]:
import torch
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(12,3))
host = fig.add_subplot(111)

for i in range(0,20):
    history = torch.load("hist" + str(i) + ".pt")
    num_evals = history["num_evals"]
    dist_history = history["dist_history"]

    plt.plot(num_evals, dist_history)

# plt.axvline(x=40, color='red', linestyle='--', label="NLBGS Converged"

host.set_title('Convergence history: intersection distance vs residual evals')
host.set_xlabel('Individual residual evals')
host.set_ylabel('Distance from OpenMDAO result')

## Note: intersection distance converging to values larger than about 1E-2 indiciates that the intersection point occurs outside the bounds of the problem. This needs to be addressed. For example:

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.utils.transforms import normalize

mt_model = result.model
y = result.train_y
bounds = sat_prob.bounds

input_vec = torch.tensor([0.4074, 0.5940, 0.8677, 0.3936, 0.0059])
npts = 40
# xvec is u12, yvec is u21
xvec, yvec = torch.meshgrid(torch.linspace(6.,12.,npts), torch.linspace(6.,20.,npts), indexing='ij')

# Analytic
sat_prob.set_vars(torch.column_stack([input_vec.repeat(1600,1),xvec.reshape(-1,1),yvec.reshape(-1,1)]))

r1vec = sat_prob.res[:,0]
r2vec = sat_prob.res[:,1]

# Predictions
test_x = torch.column_stack([input_vec.repeat(npts**2,1),
                             xvec.reshape(-1,1),
                             yvec.reshape(-1,1)])
test_x = normalize(test_x,bounds)
test_x1 = torch.column_stack([test_x, torch.ones(npts**2,1)*0]) # task id1
test_x2 = torch.column_stack([test_x, torch.ones(npts**2,1)*1]) # task id2
# test_x1.requires_grad = True

prediction1 = y.mean().item()+(mt_model.likelihood(mt_model(test_x1)))*y.std().item()
prediction2 = y.mean().item()+(mt_model.likelihood(mt_model(test_x2)))*y.std().item()

fig = plt.figure(figsize=(18,6))

# r1
ax1 = fig.add_subplot(131)
cf1 = ax1.contourf(xvec, yvec, prediction1.mean.detach().numpy().reshape(40,40),
                  cmap = 'plasma')
ax1.contour(xvec,yvec,prediction1.mean.detach().numpy().reshape(40,40), 
            levels=[0.], 
            linestyles='-', linewidths=3, colors='k')
ax1.contour(xvec, yvec, prediction1.mean.detach().reshape(40, 40) + 2.* prediction1.stddev.detach().reshape(40, 40), levels=[0.], linestyles='--', linewidths=1, colors='k')
ax1.contour(xvec, yvec, prediction1.mean.detach().reshape(40, 40) - 2.* prediction1.stddev.detach().reshape(40, 40), levels=[0.], linestyles='--', linewidths=1, colors='k')
# ax1.scatter(train_x1[...,-2],train_x1[...,-1],c='w',s=15)
fig.colorbar(cf1)
ax1.set_title(r'$r_1$')
ax1.set_xlabel(r'$u_{12}$')
ax1.set_ylabel(r'$u_{21}$')

# r2
ax2 = fig.add_subplot(132)
cf2 = ax2.contourf(xvec, yvec, prediction2.mean.detach().numpy().reshape(40,40),
                  cmap = 'plasma')
ax2.contour(xvec,yvec,prediction2.mean.detach().numpy().reshape(40,40), 
            levels=[0.], 
            linestyles='-', linewidths=3, colors='k')
ax2.contour(xvec, yvec, prediction2.mean.detach().reshape(40, 40) + 2.* prediction2.stddev.detach().reshape(40, 40), levels=[0.], linestyles='--', linewidths=1, colors='k')
ax2.contour(xvec, yvec, prediction2.mean.detach().reshape(40, 40) - 2.* prediction2.stddev.detach().reshape(40, 40), levels=[0.], linestyles='--', linewidths=1, colors='k')
# ax2.scatter(train_x2[...,-2],train_x2[...,-1],c='w',s=15)
fig.colorbar(cf2)
ax2.set_title(r'$r_2$')
ax2.set_xlabel(r'$u_{12}$')
ax2.set_ylabel(r'$u_{21}$')

# both
ax3 = fig.add_subplot(133)
# cf3 = ax3.contourf(xvec,yvec,prediction3.mean.detach().numpy().reshape(40,40),
#              cmap = 'plasma')
# fig.colorbar(cf3)
s1 = ax3.contour(xvec,yvec,r1vec.reshape(40,40),
            levels=[0.],
            linestyles='--', linewidths=3, colors='r')
s1_artist,_ = s1.legend_elements()
ax3.contour(xvec,yvec,r2vec.reshape(40,40),
            levels=[0.],
            linestyles='--', linewidths=3, colors='r')
s2 = ax3.contour(xvec,yvec,prediction1.mean.detach().numpy().reshape(40,40),
            levels=[0.],
            linestyles='-', linewidths=1.5, colors='k')
s2_artist,_ = s2.legend_elements()
s3 = ax3.contour(xvec,yvec,prediction2.mean.detach().numpy().reshape(40,40),
            levels=[0.],
            linestyles='-', linewidths=1.5, colors='b')
s3_artist,_ = s3.legend_elements()
ax3.set_title(r'$r_1$,$r_2$')
ax3.set_xlabel(r'$u_{12}$')
ax3.set_ylabel(r'$u_{21}$')
#ax3.legend([s1_artist, s3_artist], ['Analytic Residuals','Model Residuals'])
ax3.legend([s1_artist[0], s2_artist[0], s3_artist[0]],['Analytic Residuals','Model Predicted $r_1$','Model Predicted $r_2$'])